In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor

import numpy as np
from joblib import dump
import sys
import os

In [2]:
sys.path.append(os.path.abspath(".."))

from Data_Preparation.soil_moisture_data_preparation import (
    create_soil_moisture_pipeline,
    prepare_soil_moisture_data
)

In [3]:
def evaluate_xgb(X_train, y_train, X_dev, y_dev):
    print("Evaluating XGBoost Regressor for Soil Moisture...")

    param_grid = {
        'algo__n_estimators': [100, 300, 500, 1000],
        'algo__max_depth': [3, 5],
        'algo__learning_rate': [0.05, 0.1],
        'algo__subsample': [0.8, 1.0],
    }

    pipeline = create_soil_moisture_pipeline()

    pipeline_with_algo = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('algo', XGBRegressor(objective='reg:squarederror', random_state=42))
    ])

    grid_search = GridSearchCV(
        pipeline_with_algo,
        param_grid,
        cv=3,
        scoring='neg_mean_squared_error',
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_estimator = grid_search.best_estimator_

    try:
        model = best_estimator.named_steps["algo"]
        feature_names = X_train.columns
        importances = model.feature_importances_

        feature_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("\nTop 10 Most Important Features:")
        print(feature_df.head(10))

    except Exception as e:
        print("Could not extract feature importances:", e)

    y_pred = best_estimator.predict(X_dev)
    rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
    mae = mean_absolute_error(y_dev, y_pred)
    r2 = r2_score(y_dev, y_pred)

    print("Grid searching is done!")
    print("Best score (neg MSE):", grid_search.best_score_)
    print("Best hyperparameters:")
    print(grid_search.best_params_)

    return best_estimator, rmse, mae, r2

In [4]:
def evaluate_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_target = np.mean(y_true)

    print(f"\n📊 {label} Set Performance:")
    print(f"Mean of y_{label.lower()}: {mean_target:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

    return rmse, mae, r2

In [5]:
print("\n🚀 Evaluating model for: Soil Moisture")

X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_soil_moisture_data()


🚀 Evaluating model for: Soil Moisture


In [6]:
best_model, _, _, _ = evaluate_xgb(X_train, y_train, X_dev, y_dev)

Evaluating XGBoost Regressor for Soil Moisture...
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Top 10 Most Important Features:
                Feature  Importance
8   predicted_soil_temp    0.375780
7                  year    0.120229
0              latitude    0.101498
5                 month    0.098748
2       air_temperature    0.090009
4   total_precipitation    0.073458
3  dewpoint_temperature    0.070362
1             longitude    0.035543
6                   day    0.034372
Grid searching is done!
Best score (neg MSE): -0.002317330816814262
Best hyperparameters:
{'algo__learning_rate': 0.1, 'algo__max_depth': 5, 'algo__n_estimators': 1000, 'algo__subsample': 0.8}


In [7]:
print("✅ Data Split Shapes:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)
print("  X_test:", X_test.shape)
print("  y_train:", y_train.shape)
print("  y_dev:", y_dev.shape)
print("  y_test:", y_test.shape)

✅ Data Split Shapes:
  X_train: (1679864, 9)
  X_dev: (419967, 9)
  X_test: (10000, 9)
  y_train: (1679864,)
  y_dev: (419967,)
  y_test: (10000,)


In [8]:
y_train_pred = best_model.predict(X_train)
y_dev_pred = best_model.predict(X_dev)
y_test_pred = best_model.predict(X_test)

In [9]:
train_rmse, train_mae, train_r2 = evaluate_metrics(y_train, y_train_pred, "Train")
dev_rmse, dev_mae, dev_r2 = evaluate_metrics(y_dev, y_dev_pred, "Dev")
test_rmse, test_mae, test_r2 = evaluate_metrics(y_test, y_test_pred, "Test")


📊 Train Set Performance:
Mean of y_train: 0.1797
RMSE: 0.0479
MAE: 0.0351
R²: 0.6878

📊 Dev Set Performance:
Mean of y_dev: 0.1796
RMSE: 0.0482
MAE: 0.0354
R²: 0.6829

📊 Test Set Performance:
Mean of y_test: 0.1770
RMSE: 0.0470
MAE: 0.0347
R²: 0.6906


In [12]:
dump(best_model, "../models/soil_moisture_model.joblib")

print("✅ Model saved as: ../models/soil_moisture_model.joblib")

✅ Model saved as: ../models/soil_moisture_model.joblib


In [13]:
results_df = pd.DataFrame({
    "actual_soil_moisture": y_test.values,
    "predicted_soil_moisture": y_test_pred
})

results_df.head(10)

,actual_soil_moisture,predicted_soil_moisture
0,0.178049,0.167806
1,0.159784,0.174516
2,0.189298,0.191272
3,0.149692,0.139895
4,0.141275,0.139288
5,0.055885,0.052839
6,0.156202,0.145669
7,0.256181,0.240013
8,0.119157,0.165278
9,0.125611,0.127958


In [ ]:
# def evaluate_rf(X_train, y_train, X_dev, y_dev):
#     print("Evaluating Random Forest Regressor...")

#     param_grid = {
#         'algo__n_estimators': [100, 300],
#         'algo__max_depth': [5, 10, None],
#         'algo__min_samples_split': [2, 5],
#         'algo__min_samples_leaf': [1, 2],
#     }

#     pipeline = create_soil_moisture_pipeline() 

#     pipeline_with_algo = Pipeline(steps=[
#         ('preprocessor', pipeline),
#         ('algo', RandomForestRegressor(random_state=42, n_jobs=-1))
#     ])

#     grid_search = GridSearchCV(
#         pipeline_with_algo,
#         param_grid,
#         cv=3,
#         scoring='neg_mean_squared_error',
#         verbose=1,
#         n_jobs=-1
#     )

#     grid_search.fit(X_train, y_train)
#     best_estimator = grid_search.best_estimator_

#     try:
#         model = best_estimator.named_steps["algo"]
#         feature_names = X_train.columns
#         importances = model.feature_importances_

#         feature_df = pd.DataFrame({
#             "Feature": feature_names,
#             "Importance": importances
#         }).sort_values(by="Importance", ascending=False)

#         print("\nTop 10 Most Important Features:")
#         print(feature_df.head(10))

#     except Exception as e:
#         print("Could not extract feature importances:", e)

#     y_pred = best_estimator.predict(X_dev)
#     rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
#     mae = mean_absolute_error(y_dev, y_pred)
#     r2 = r2_score(y_dev, y_pred)

#     print("Grid searching is done!")
#     print("Best score (neg MSE):", grid_search.best_score_)
#     print("Best hyperparameters:")
#     print(grid_search.best_params_)

#     return best_estimator, rmse, mae, r2

In [ ]:
# print("\n🚀 Evaluating model for: Soil Moisture")

# X_train, X_dev, X_test, y_train, y_dev, y_test = prepare_soil_moisture_data()


🚀 Evaluating model for: Soil Moisture


In [ ]:
# best_model, _, _, _ = evaluate_rf(X_train, y_train, X_dev, y_dev)

Evaluating Random Forest Regressor...
Fitting 3 folds for each of 24 candidates, totalling 72 fits


Python(41490) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41491) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41492) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41493) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41494) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41495) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41496) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(41497) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:
# print("✅ Data Split Shapes:")
# print("  X_train:", X_train.shape)
# print("  X_dev:", X_dev.shape)
# print("  X_test:", X_test.shape)
# print("  y_train:", y_train.shape)
# print("  y_dev:", y_dev.shape)
# print("  y_test:", y_test.shape)

In [ ]:
# y_train_pred = best_model.predict(X_train)
# y_dev_pred = best_model.predict(X_dev)
# y_test_pred = best_model.predict(X_test)

In [ ]:
# train_rmse, train_mae, train_r2 = evaluate_metrics(y_train, y_train_pred, "Train")
# dev_rmse, dev_mae, dev_r2 = evaluate_metrics(y_dev, y_dev_pred, "Dev")
# test_rmse, test_mae, test_r2 = evaluate_metrics(y_test, y_test_pred, "Test")